# TabPFN vs. the tabular ladder, and with the opposition signal

**Runs in Google Colab on a GPU** (Runtime → Change runtime type → **T4 GPU**). About 15–25 minutes in total.

**The question, in course terms:** TabPFN is a transformer pre-trained on millions of synthetic tables (Lecture 12's pre-training idea, applied to tables). Does it beat the tuned models, against Lecture 8's finding that tree ensembles usually win on tabular data?

**What it runs (validation only; the 2019–2020 test cohort is never scored):**
1. `pipeline`: historical rate, LASSO, gradient boosting and TabPFN. Each model gets **rolling CV** (2010–2014) and **out-of-time** (fit 2005–2014, score 2015–2018) for its selected configuration.
2. `stack`: TabPFN as the first stage, then the Sabin opposition flag as a second stage on 2015–2018. LASSO and gradient boosting run too, for comparison in the same environment.

**One-time setup, under Secrets (🔑) in the left sidebar:**
- `GITHUB_TOKEN`: a GitHub personal access token with read access to the private repo.
- `TABPFN_TOKEN`: from https://ux.priorlabs.ai, after accepting the model license (non-commercial use; fine for a class project).

Blank fields are imputed before TabPFN sees them (no missing-value flags), matching the Nov 15 leakage fix.

**When it finishes:** File → Download → Download .ipynb, then save it to `notebooks/` so the results can be read and logged.

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> T4 GPU, then rerun" 

In [ ]:
from google.colab import userdata
import os, subprocess
token = userdata.get("GITHUB_TOKEN")
if not os.path.exists("queue-risk-ai"):
    subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/devault-settings/queue-risk-ai.git"], check=True)
%cd queue-risk-ai
!git pull -q && git log --oneline -1

In [ ]:
!pip install -q -e . tabpfn
!apt-get -qq update > /dev/null && apt-get -qq install -y poppler-utils > /dev/null   # pdftotext, for the Sabin reports
!which pdftotext || echo "pdftotext MISSING: the Sabin step will fail"
os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")
import tabpfn, torch; print("tabpfn", tabpfn.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
!python tests/selftest.py | tail -3

## 1. The model ladder with TabPFN

Each line prints the rolling-CV AUC and the out-of-time AUC on 2015–2018 for the **same** selected configuration. The reference numbers from Nov 15 (CPU, no TabPFN) are LASSO **0.631 / 0.686** and gradient boosting **0.663 / 0.633**.

In [ ]:
!python -W ignore -m queue_risk.pipeline --models "Historical,LASSO,Gradient,TabPFN" 

In [ ]:
import pandas as pd
pd.read_csv("reports/results_within_validation.csv").drop(columns="secs").round(3)

## 2. TabPFN as the first stage, plus the opposition flag

The first command downloads the five Sabin editions from the Internet Archive and parses them (a few minutes). The stacking check then fits the second stage leave-one-year-out on 2015–2018.

**If part 1 already ran in this session,** you can skip it: run the install cell (3), then the cells from here down.

In [ ]:
!rm -f data/external/sabin/*.txt   # drop any half-made text files from a failed run
!python -W ignore -m queue_risk.sabin | tail -3
!ls data/interim/

In [ ]:
!python -W ignore -m queue_risk.stack --stage1 tabpfn

In [ ]:
!python -W ignore -m queue_risk.stack --stage1 both    # LASSO and gradient boosting, same environment, for comparison

Done. **File → Download → Download .ipynb** and save it as `notebooks/tabpfn_colab.ipynb`, replacing this copy, so the outputs can be read and logged in `docs/model_log.md`.